# Silver Layer - Part 2.A: Transformations

Dataset: French Road Safety Open Data, year 2024.

This notebook applies the Silver layer transformations identified from the
Part 1 profiling and data quality analysis: standardization, cleaning,
enrichment and deduplication.

In [1]:
import pandas as pd
from pathlib import Path

BRONZE_DIR = Path("data/bronze")
SILVER_DIR = Path("data/silver")

LAT_MIN, LAT_MAX = -22, 51
LON_MIN, LON_MAX = -63, 56

NEGATIVE_ONE_COLUMNS = {
    "caract": ["col"],
    "lieux": ["circ", "vosp", "prof", "plan", "surf", "infra", "situ"],
    "vehicules": ["senc", "choc", "motor"],
    "usagers": ["place", "sexe", "etatp"],
}

## Load the raw tables

In [2]:
caract = pd.read_csv(BRONZE_DIR / "caract-2024.csv", sep=";", low_memory=False)
lieux = pd.read_csv(BRONZE_DIR / "lieux-2024.csv", sep=";", low_memory=False)
vehicules = pd.read_csv(BRONZE_DIR / "vehicules-2024.csv", sep=";", low_memory=False)
usagers = pd.read_csv(BRONZE_DIR / "usagers-2024.csv", sep=";", low_memory=False)

print(caract.shape, lieux.shape, vehicules.shape, usagers.shape)

(54402, 15) (70248, 18) (92678, 11) (125187, 16)


## Standardization and cleaning

- Coordinates converted from comma-decimal strings to float degrees, and set to
  missing when outside a plausible France bounding box.
- Date fields combined into a single `accident_date` column, plus an hour and
  time-of-day category derived from `hrmn`.
- Codes equal to -1 ("not specified") replaced with missing values.

In [3]:
def replace_not_specified(df, columns):
    for col in columns:
        df[col] = df[col].replace(-1, pd.NA)
    return df

def time_of_day_category(hour):
    if 5 <= hour <= 11:
        return "Morning"
    if 12 <= hour <= 17:
        return "Afternoon"
    if 18 <= hour <= 21:
        return "Evening"
    return "Night"

caract = caract.drop_duplicates()
caract = caract.drop_duplicates(subset="Num_Acc")

caract["lat"] = pd.to_numeric(caract["lat"].astype(str).str.replace(",", "."), errors="coerce")
caract["long"] = pd.to_numeric(caract["long"].astype(str).str.replace(",", "."), errors="coerce")

out_of_range = ~caract["lat"].between(LAT_MIN, LAT_MAX) | ~caract["long"].between(LON_MIN, LON_MAX)
caract.loc[out_of_range, ["lat", "long"]] = pd.NA

caract["hour"] = caract["hrmn"].str.slice(0, 2).astype(int)
caract["time_of_day"] = caract["hour"].apply(time_of_day_category)

caract["accident_date"] = pd.to_datetime(
    dict(year=caract["an"], month=caract["mois"], day=caract["jour"]),
    errors="coerce",
)
caract["weekday"] = caract["accident_date"].dt.day_name()

caract = replace_not_specified(caract, NEGATIVE_ONE_COLUMNS["caract"])

caract.head()

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,atm,col,adr,lat,long,hour,time_of_day,accident_date,weekday
0,202400000001,25,3,2024,07:40,2,70,70285,1,1,5,1,D438,47.562770,6.758320,7,Morning,2024-03-25,Monday
1,202400000002,20,3,2024,15:05,1,21,21054,2,3,7,6,HOTEL DIEU (RUE DE L'),47.021090,4.837550,15,Afternoon,2024-03-20,Wednesday
2,202400000003,22,3,2024,19:30,2,15,15012,1,1,1,6,Allée des Tilleuls,44.902384,2.496418,19,Evening,2024-03-22,Friday
3,202400000004,24,3,2024,17:50,1,14,14118,2,3,7,3,128 Rue d'Authie,49.191660,-0.398510,17,Afternoon,2024-03-24,Sunday
4,202400000005,25,3,2024,19:35,5,13,13106,1,3,2,5,BEDOULE (CHEMIN DE LA),43.390000,5.350000,19,Evening,2024-03-25,Monday


In [4]:
lieux = lieux.drop_duplicates()
lieux = lieux.drop_duplicates(subset="Num_Acc")
lieux = replace_not_specified(lieux, NEGATIVE_ONE_COLUMNS["lieux"])

lieux.head()

,Num_Acc,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
0,202400000001,3,D438,0,NaN,2,2,0,1,1,260,2,NaN,7,1,0,1,90
1,202400000002,4,HOTEL DIEU (RUE DE L'),0,NaN,2,2,0,1,-1,-1,1,NaN,-1,9,0,1,30
3,202400000003,4,TILLEULS (ALLEE DES),0,NaN,2,2,0,1,-1,-1,1,NaN,-1,1,0,3,50
4,202400000004,4,AUTHIE (N° 106 PAIRS -115 IMPAIRS),0,NaN,2,4,0,1,-1,-1,1,NaN,-1,1,9,1,50
6,202400000005,1,BEDOULE (CHEMIN DE LA),0,NaN,2,4,0,1,1,0,1,NaN,17,2,0,1,50


In [5]:
vehicules = vehicules.drop_duplicates()
vehicules = replace_not_specified(vehicules, NEGATIVE_ONE_COLUMNS["vehicules"])

vehicules.head()

,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
0,202400000001,155 781 758,A01,1,7,0,2,1,13,1,NaN
1,202400000001,155 781 759,B01,2,14,0,2,2,21,1,NaN
2,202400000002,155 781 757,A01,1,10,0,1,3,15,1,NaN
3,202400000003,155 781 756,A01,3,3,9,1,1,1,1,NaN
4,202400000004,155 781 754,B01,3,34,0,2,0,0,1,NaN


In [6]:
usagers = usagers.drop_duplicates()
usagers = replace_not_specified(usagers, NEGATIVE_ONE_COLUMNS["usagers"])

usagers["an_nais"] = pd.to_numeric(usagers["an_nais"], errors="coerce")
usagers["age"] = 2024 - usagers["an_nais"]
invalid_age = (usagers["age"] < 0) | (usagers["age"] > 110)
usagers.loc[invalid_age, "age"] = pd.NA

usagers.head()

,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp,age
0,202400000001,203 988 581,155 781 758,A01,1,1,3,1,2003.0,2,1,-1,-1,-1,-1,<NA>,21.0
1,202400000001,203 988 582,155 781 759,B01,1,1,1,1,1997.0,4,1,-1,-1,-1,-1,<NA>,27.0
2,202400000002,203 988 579,155 781 757,A01,10,3,3,2,1927.0,5,0,-1,-1,3,3,1,97.0
3,202400000002,203 988 580,155 781 757,A01,1,1,1,1,1987.0,4,1,0,-1,3,3,1,37.0
4,202400000003,203 988 574,155 781 756,A01,2,2,4,2,2007.0,5,8,0,-1,-1,-1,<NA>,17.0


## Enrichment

- `severity_index` and `severity_category` per accident, computed from the
  worst injuries recorded among the users involved.
- `nb_vehicles` per accident.

In [7]:
severity = usagers.groupby("Num_Acc")["grav"].agg(
    nb_users="count",
    nb_killed=lambda s: (s == 2).sum(),
    nb_hospitalized=lambda s: (s == 3).sum(),
    nb_slightly_injured=lambda s: (s == 4).sum(),
    nb_unharmed=lambda s: (s == 1).sum(),
).reset_index()

severity["severity_index"] = (
    3 * severity["nb_killed"]
    + 2 * severity["nb_hospitalized"]
    + 1 * severity["nb_slightly_injured"]
)

def severity_category(row):
    if row["nb_killed"] > 0:
        return "Fatal"
    if row["nb_hospitalized"] > 0:
        return "Severe"
    if row["nb_slightly_injured"] > 0:
        return "Minor"
    return "No injury"

severity["severity_category"] = severity.apply(severity_category, axis=1)

vehicle_counts = vehicules.groupby("Num_Acc").size().reset_index(name="nb_vehicles")

severity.head()

,Num_Acc,nb_users,nb_killed,nb_hospitalized,nb_slightly_injured,nb_unharmed,severity_index,severity_category
0,202400000001,2,0,1,0,1,2,Severe
1,202400000002,2,0,1,0,1,2,Severe
2,202400000003,5,0,1,3,1,5,Severe
3,202400000004,1,0,0,1,0,1,Minor
4,202400000005,4,0,0,3,1,3,Minor


## Deduplication and merge

`lieux` is not unique per `Num_Acc` in the source data, so only the first
record per accident is kept before merging, to preserve the accident grain.

In [8]:
accidents_silver = caract.merge(lieux, on="Num_Acc", how="left")
accidents_silver = accidents_silver.merge(severity, on="Num_Acc", how="left")
accidents_silver = accidents_silver.merge(vehicle_counts, on="Num_Acc", how="left")

accidents_silver["nb_users"] = accidents_silver["nb_users"].fillna(0).astype(int)
accidents_silver["nb_vehicles"] = accidents_silver["nb_vehicles"].fillna(0).astype(int)
accidents_silver["severity_index"] = accidents_silver["severity_index"].fillna(0).astype(int)
accidents_silver["severity_category"] = accidents_silver["severity_category"].fillna("No injury")

accidents_silver.head()

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,...,situ,vma,nb_users,nb_killed,nb_hospitalized,nb_slightly_injured,nb_unharmed,severity_index,severity_category,nb_vehicles
0,202400000001,25,3,2024,07:40,2,70,70285,1,1,...,1,90,2,0,1,0,1,2,Severe,2
1,202400000002,20,3,2024,15:05,1,21,21054,2,3,...,1,30,2,0,1,0,1,2,Severe,1
2,202400000003,22,3,2024,19:30,2,15,15012,1,1,...,3,50,5,0,1,3,1,5,Severe,1
3,202400000004,24,3,2024,17:50,1,14,14118,2,3,...,1,50,1,0,0,1,0,1,Minor,2
4,202400000005,25,3,2024,19:35,5,13,13106,1,3,...,1,50,4,0,0,3,1,3,Minor,4


## Save the Silver tables

In [9]:
SILVER_DIR.mkdir(parents=True, exist_ok=True)
accidents_silver.to_csv(SILVER_DIR / "accidents_silver.csv", index=False)
vehicules.to_csv(SILVER_DIR / "vehicules_silver.csv", index=False)
usagers.to_csv(SILVER_DIR / "usagers_silver.csv", index=False)

print(f"Silver accidents table: {len(accidents_silver)} rows")
print(f"Silver vehicules table: {len(vehicules)} rows")
print(f"Silver usagers table: {len(usagers)} rows")

Silver accidents table: 54402 rows
Silver vehicules table: 92678 rows
Silver usagers table: 125187 rows


## Recap: what changed and why

**Standardization** - `lat`/`long` went from comma-decimal strings to actual
float degrees, `jour`/`mois`/`an` were combined into one `accident_date`
column, and `hrmn` was split into an `hour` plus a `time_of_day` category
(Morning, Afternoon, Evening, Night) so the accident time can be grouped and
filtered instead of parsed as text every time.

**Cleaning** - coordinates outside a plausible France bounding box
(lat in [-22, 51], long in [-63, 56], to cover both mainland France and the
overseas departments) were set to missing rather than kept as-is or dropped
entirely, since the rest of the accident record is still usable even without
a reliable location. The `-1` "not specified" code was recoded to a proper
missing value in every column where it showed up as a false category during
the data quality check (`circ`, `vosp`, `prof`, `plan`, `surf`, `infra`,
`situ`, `senc`, `choc`, `motor`, `place`, `sexe`, `col`). User age is
computed as accident year minus birth year, with negative or unrealistic
ages (over 110) set to missing instead of silently kept.

**Enrichment** - two fields did not exist in the raw data and had to be
derived: `severity_index` (3 x killed + 2 x hospitalized + 1 x slightly
injured, per accident) and `severity_category` (Fatal, Severe, Minor or No
injury, based on the worst injury among the users involved). `nb_users` and
`nb_vehicles` are also aggregated onto the accident row, since neither the
usagers nor vehicules table stores that count directly.

**Deduplication** - full duplicate rows were dropped from all four tables.
The more interesting case is `lieux`: it turned out not to be unique per
`Num_Acc` in the raw data, even though one row per accident is what the
documentation implies. Only the first record per accident is kept before
merging, otherwise the join below would have silently multiplied some
accidents.